In [ ]:
import os
import pandas as pd
import numpy as np
from redcap import Project

REDCAP_API_URL = os.getenv("AIM8_REDCAP_API_URL", "https://redcap.research.yale.edu/api/")
RELAUNCH_API_TOKEN = "64C5D967CBEB77335224862283A74F4D"

project = Project(REDCAP_API_URL, RELAUNCH_API_TOKEN)
raw_df = project.export_records(format_type="df").reset_index(names="record_id")

print(f"Exported {len(raw_df)} records, {len(raw_df.columns)} fields")
raw_df.head(3)

In [ ]:
# ── Set the record ID you want to debug ──────────────────────────────────────
RECORD_ID = None  # <-- replace with the actual record_id (int or str)
# ─────────────────────────────────────────────────────────────────────────────

df = raw_df.copy()
df["record_id"] = pd.to_numeric(df["record_id"], errors="coerce")
row = df.loc[df["record_id"] == int(RECORD_ID)]
if row.empty:
    raise ValueError(f"record_id {RECORD_ID!r} not found in export")
row = row.iloc[0]

def val(field):
    """Return raw value or NaN; coerces to numeric where possible."""
    if field not in row.index:
        return "FIELD MISSING FROM EXPORT"
    v = row[field]
    if pd.isna(v):
        return None
    num = pd.to_numeric(v, errors="coerce")
    return num if not pd.isna(num) else str(v)

def numval(field):
    """Return numeric value or NaN."""
    v = val(field)
    if v == "FIELD MISSING FROM EXPORT" or v is None:
        return np.nan
    try:
        return float(v)
    except (TypeError, ValueError):
        return np.nan

# REDCap checkbox fields: interested_spstudy_consent(1) → interested_spstudy_consent___1
def consent(n):
    return val(f"interested_spstudy_consent___{n}")

print("=" * 70)
print(f"DEBUG: record_id = {RECORD_ID}")
print("=" * 70)

# ── Raw field values ─────────────────────────────────────────────────────────
fields_to_show = [
    "age_v2",
    "cognition_screener_v2",
    "seizure_hx_v2",
    "intox_screen_v2",
    "no_computer",
    "english_fluency",
    "interested_spstudy_consent___1",
    "interested_spstudy_consent___2",
    "interested_spstudy_consent___3",
    "interested_spstudy_consent___4",
    "atypical_recentuse",
    "raven_total_score_v2",
    "planning_trip",
    "geo_crit",
    "sp_dayslastuse",
]

print("\n--- Raw field values ---")
for f in fields_to_show:
    print(f"  {f:<40s} = {val(f)!r}")

# ── Expression 1: ineligibility logic (should be True if ineligible) ─────────
print("\n" + "=" * 70)
print("EXPRESSION 1: ineligibility logic")
print("  (True = ineligible for that reason; ANY True → record is ineligible)")
print("=" * 70)

age = numval("age_v2")

conds_1 = [
    ("[age_v2] > 65",                          age > 65),
    ("[age_v2] < 18",                          age < 18),
    ("[cognition_screener_v2] = '1'",           val("cognition_screener_v2") == 1),
    ("[seizure_hx_v2] = '1'",                  val("seizure_hx_v2") == 1),
    ("[intox_screen_v2] = '1'",                val("intox_screen_v2") == 1),
    ("[no_computer] = '1'",                    val("no_computer") == 1),
    ("[english_fluency] = '0'",                val("english_fluency") == 0),
    ("[interested_spstudy_consent(1)] = 0",    consent(1) == 0),
    ("[interested_spstudy_consent(2)] = 0",    consent(2) == 0),
    ("[interested_spstudy_consent(3)] = 0",    consent(3) == 0),
    ("[interested_spstudy_consent(4)] = 0",    consent(4) == 0),
    ("[atypical_recentuse] = '1'",             val("atypical_recentuse") == 1),
    ("[raven_total_score_v2] < 1",             numval("raven_total_score_v2") < 1),
    ("[planning_trip] = '0'",                  val("planning_trip") == 0),
    ("[geo_crit] = ''",                        val("geo_crit") in (None, "", "FIELD MISSING FROM EXPORT")),
]

any_true_1 = False
for label, result in conds_1:
    flag = ">>> TRUE (ineligible reason)" if result else "false"
    print(f"  {label:<50s}: {flag}")
    if result:
        any_true_1 = True

print(f"\n  Overall Expression 1 (ineligible): {any_true_1}")

# ── Expression 2: eligibility logic (should be True if eligible) ─────────────
print("\n" + "=" * 70)
print("EXPRESSION 2: eligibility logic")
print("  (ALL must be True for the record to be eligible)")
print("=" * 70)

conds_2 = [
    ("[age_v2] < 65",                          age < 65),
    ("[age_v2] > 17",                          age > 17),
    ("[cognition_screener_v2] = '0'",           val("cognition_screener_v2") == 0),
    ("[seizure_hx_v2] = '0'",                  val("seizure_hx_v2") == 0),
    ("[intox_screen_v2] = '0'",                val("intox_screen_v2") == 0),
    ("[no_computer] = '0'",                    val("no_computer") == 0),
    ("[geo_crit] <> ''",                       val("geo_crit") not in (None, "", "FIELD MISSING FROM EXPORT")),
    ("[planning_trip] = '1'",                  val("planning_trip") == 1),
    ("[interested_spstudy_consent(1)] = '1'",  consent(1) == 1),
    ("[interested_spstudy_consent(2)] = '1'",  consent(2) == 1),
    ("[interested_spstudy_consent(3)] = '1'",  consent(3) == 1),
    ("[interested_spstudy_consent(4)] = '1'",  consent(4) == 1),
    ("[atypical_recentuse] = '0'",             val("atypical_recentuse") == 0),
    ("[sp_dayslastuse] > 41",                  numval("sp_dayslastuse") > 41),
    ("[raven_total_score_v2] > 0",             numval("raven_total_score_v2") > 0),
]

all_true_2 = True
for label, result in conds_2:
    flag = "true" if result else ">>> FALSE (blocking eligibility)"
    print(f"  {label:<50s}: {flag}")
    if not result:
        all_true_2 = False

print(f"\n  Overall Expression 2 (eligible): {all_true_2}")